# Generate `ground_truth_context.json` (self-contained)

Builds a ground-truth JSON with the **same schema** as `pdfs_test/ground_truth_context.json` for **any folder of PDFs**.

Everything lives in this notebook (no imports from other project scripts).

| Field | How |
|-------|-----|
| `caption` | Heuristics on LightOnOCR text (`Table N: …`) |
| `mentions` | Heuristics: paragraphs citing `Table N` |
| `datasets` / `metrics` | GLiNER2 on table HTML only |
| `table_id` | `{pdf_stem}_p{page}_t{n}` |

No PwC, no Ollama, no overrides.

**Note:** this is an auto draft. `pdfs_test` was manually reviewed — edit the JSON before treating it as gold.

Run from `table_extraction/`.


In [ ]:
# Optional on a clean server:
# !pip install -q torch "transformers>=4.57.6" pypdfium2 pillow beautifulsoup4 "gliner2>=1.2.5" pylatexenc

import json
import os
import re
import tempfile
import unicodedata
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pypdfium2 as pdfium
import torch
from bs4 import BeautifulSoup
from PIL import Image

print("Imports OK")


In [ ]:
# --- Config: any folder of PDFs ---
PDF_DIR = Path("pdfs_test")  # e.g. Path("../data/pdf_files_3")
OUTPUT_PATH = PDF_DIR / "ground_truth_context.json"
OCR_CACHE_DIR = PDF_DIR / "ocr_cache"
FORCE_OCR = False

OCR_MODEL_ID = "lightonai/LightOnOCR-2-1B"
OCR_MAX_NEW_TOKENS = 8192
OCR_TARGET_LONGEST = 1540

GLINER2_MODEL_ID = "fastino/gliner2-base-v1"
GLINER2_MIN_SCORE = 0.65
GLINER2_MAX_CHARS = 3000

ADJACENT_PARA_WINDOW = 2
MIN_MENTION_CHARS = 40

PDF_DIR.mkdir(parents=True, exist_ok=True)
OCR_CACHE_DIR.mkdir(parents=True, exist_ok=True)

PDF_FILES = sorted(PDF_DIR.glob("*.pdf"))
assert PDF_FILES, f"No PDFs in {PDF_DIR.resolve()}"

print(f"PDF_DIR:   {PDF_DIR.resolve()}")
print(f"OUTPUT:    {OUTPUT_PATH.resolve()}")
print(f"OCR cache: {OCR_CACHE_DIR.resolve()}")
print(f"PDFs: {len(PDF_FILES)}")
for p in PDF_FILES:
    print(f"  - {p.name}")


In [ ]:
# --- Load models ---
from gliner2 import GLiNER2
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor

os.environ.setdefault("PYTORCH_JIT", "0")
os.environ.setdefault("PYTORCH_NVFUSER_DISABLE", "1")
for name, args in [
    ("_jit_set_profiling_executor", (False,)),
    ("_jit_set_profiling_mode", (False,)),
    ("_jit_override_can_fuse_on_gpu", (False,)),
    ("_jit_override_can_fuse_on_cpu", (False,)),
    ("_jit_set_texpr_fuser_enabled", (False,)),
    ("_jit_set_nvfuser_enabled", (False,)),
]:
    fn = getattr(torch._C, name, None)
    if fn is not None:
        try:
            fn(*args)
        except Exception:
            pass

if torch.cuda.is_available():
    ocr_device = "cuda"
    ocr_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif torch.backends.mps.is_available():
    ocr_device = "mps"
    ocr_dtype = torch.float16
else:
    ocr_device = "cpu"
    ocr_dtype = torch.float32

print(f"OCR device: {ocr_device} | dtype: {ocr_dtype}")
ocr_processor = LightOnOcrProcessor.from_pretrained(OCR_MODEL_ID)
ocr_model = LightOnOcrForConditionalGeneration.from_pretrained(
    OCR_MODEL_ID,
    torch_dtype=ocr_dtype,
    attn_implementation="eager",
).to(ocr_device)

gliner2_map = "cuda" if torch.cuda.is_available() else "cpu"
gliner2_model = GLiNER2.from_pretrained(GLINER2_MODEL_ID, map_location=gliner2_map)
print(f"GLiNER2 loaded on {gliner2_map}")


In [ ]:
# --- OCR helpers ---

def render_pdf_page(pdf_doc, page_idx: int, target_longest: int = OCR_TARGET_LONGEST) -> Image.Image:
    page = pdf_doc[page_idx]
    img = page.render(scale=200 / 72).to_pil()
    w, h = img.size
    longest = max(w, h)
    if longest > target_longest:
        ratio = target_longest / longest
        img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    return img.convert("RGB") if img.mode != "RGB" else img


def ocr_page(img: Image.Image, max_new_tokens: int = OCR_MAX_NEW_TOKENS) -> str:
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    img.save(tmp, format="PNG")
    tmp.close()
    try:
        conv = [{"role": "user", "content": [{"type": "image", "url": tmp.name}]}]
        inputs = ocr_processor.apply_chat_template(
            conv, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
        )
        inputs = {
            k: v.to(device=ocr_device, dtype=ocr_dtype) if v.is_floating_point() else v.to(ocr_device)
            for k, v in inputs.items()
        }
        with torch.no_grad():
            out = ocr_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gen = out[0, inputs["input_ids"].shape[1]:]
        return ocr_processor.decode(gen, skip_special_tokens=True)
    finally:
        os.unlink(tmp.name)


def ocr_pdf_pages(pdf_path: Path, *, force_ocr: bool = False) -> List[str]:
    OCR_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_file = OCR_CACHE_DIR / f"{pdf_path.stem}_pages.json"
    if cache_file.exists() and not force_ocr:
        with cache_file.open(encoding="utf-8") as f:
            return json.load(f)["pages"]

    pdf_doc = pdfium.PdfDocument(str(pdf_path))
    pages: List[str] = []
    try:
        for page_idx in range(len(pdf_doc)):
            print(f"  OCR page {page_idx + 1}/{len(pdf_doc)}", end="\r", flush=True)
            pages.append(ocr_page(render_pdf_page(pdf_doc, page_idx)))
    finally:
        pdf_doc.close()
    print()
    with cache_file.open("w", encoding="utf-8") as f:
        json.dump({"file_name": str(pdf_path), "pages": pages}, f, ensure_ascii=False)
    return pages

print("OCR helpers ready")


In [ ]:
# --- Caption + mentions heuristics ---

_NUM = r"[A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+"
TABLE_BLOCK_RE = re.compile(r"<table\b[^>]*>.*?</table>", re.DOTALL | re.IGNORECASE)
CAPTION_RE = re.compile(
    rf"(?:\*\*)?(?:Table|Tab\.?|TABLE)\s+({_NUM})(?:\*\*)?\s*[:.\u2014-]\s+",
    re.IGNORECASE,
)
TABLE_REF_RE = re.compile(
    rf"\b(?:Table|Tab\.?|TABLE|Tables|TABLES)\s+({_NUM})(?:\s*(?:,|and|&)\s*({_NUM}))*",
    re.IGNORECASE,
)
_REF_KEYWORD_RE = re.compile(r"^(?:Tables?|Tab\.?|TABLES?)\s*", re.IGNORECASE)
_REF_NUM_RE = re.compile(r"[A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+")
_FOOTNOTE_URL_RE = re.compile(r"https?://|github\.com|\$\^\{\d+\}")
_HEADING_ONLY_RE = re.compile(r"^#{1,6}\s+\S")
_HYPHEN_CONTINUATION_PREFIX_RE = re.compile(r"^([a-z][a-z'-]{0,30})([.,;:!?])?")
_SENTENCE_END_RE = re.compile(r"""[.!?]['")\]]*\s*$""")
MAX_MENTION_EXTENSIONS = 2


def normalize_table_number(raw: str) -> str:
    return raw.strip().upper().replace(" ", "")


def _collapse_ws(text: str) -> str:
    return re.sub(r"[ \t]+", " ", text).strip()


def _repair_hyphen_breaks(text: str) -> str:
    return re.sub(r"(\w)-\s*\n\s*([a-z][\w'-]*)", r"\1\2", text)


def _split_paragraphs(text: str) -> List[str]:
    text = _repair_hyphen_breaks(text)
    paras = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if len(paras) > 1:
        return [_collapse_ws(p) for p in paras]
    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    if len(lines) <= 1:
        return [_collapse_ws(text)] if text.strip() else []
    chunks, buf = [], []
    for line in lines:
        buf.append(line)
        if line.endswith((".", "!", "?")):
            chunks.append(" ".join(buf))
            buf = []
    if buf:
        chunks.append(" ".join(buf))
    if len(chunks) > 1:
        return [_collapse_ws(p) for p in chunks if p.strip()]
    return [_collapse_ws(text)] if text.strip() else []


def _table_nums_in_para(para: str) -> set:
    nums = set()
    for m in TABLE_REF_RE.finditer(para):
        body = _REF_KEYWORD_RE.sub("", m.group(0))
        for g in _REF_NUM_RE.findall(body):
            nums.add(normalize_table_number(g))
    return nums


def find_page_captions(page_text: str) -> List[dict]:
    captions = []
    for m in CAPTION_RE.finditer(page_text):
        start = m.start()
        rest = page_text[start:]
        para = re.split(r"\n\s*\n", rest, maxsplit=1)[0]
        caption = _collapse_ws(para)
        if not caption:
            continue
        captions.append({
            "table_num": normalize_table_number(m.group(1)),
            "caption": caption,
            "start": start,
            "end": start + len(para),
        })
    return captions


def pair_captions_to_tables(page_text: str) -> List[dict]:
    blocks = list(TABLE_BLOCK_RE.finditer(page_text))
    captions = find_page_captions(page_text)
    if not blocks:
        return []
    if len(captions) == len(blocks):
        caps_sorted = sorted(captions, key=lambda c: c["start"])
        return [
            {"match": bm, "caption": cap["caption"], "table_num": cap["table_num"]}
            for bm, cap in zip(blocks, caps_sorted)
        ]
    used = set()
    results = []
    for bm in blocks:
        t_start, t_end = bm.start(), bm.end()
        best_idx, best_dist = None, float("inf")
        for ci, cap in enumerate(captions):
            if ci in used:
                continue
            if cap["end"] <= t_start:
                dist = t_start - cap["end"]
            elif cap["start"] >= t_end:
                dist = cap["start"] - t_end
            else:
                dist = 0
            if dist < best_dist:
                best_dist, best_idx = dist, ci
        if best_idx is not None:
            used.add(best_idx)
            cap = captions[best_idx]
            results.append({"match": bm, "caption": cap["caption"], "table_num": cap["table_num"]})
        else:
            results.append({"match": bm, "caption": "", "table_num": None})
    return results


def page_paragraphs_without_tables(page_text: str) -> List[str]:
    return _split_paragraphs(TABLE_BLOCK_RE.sub(" ", page_text))


def _is_safe_continuation(para: str) -> bool:
    t = para.strip()
    if not t or CAPTION_RE.match(t) or re.match(r"^Table\b", t, re.IGNORECASE):
        return False
    return True


def _ends_complete_sentence(text: str) -> bool:
    return bool(_SENTENCE_END_RE.search(text.rstrip()))


def _continuation_paragraphs(page_idx, para_idx, paras, pages):
    for k in range(para_idx + 1, len(paras)):
        yield paras[k]
    if page_idx < len(pages):
        for p in page_paragraphs_without_tables(pages[page_idx]):
            yield p


def _extract_hyphen_continuation_prefix(para: str) -> Optional[str]:
    t = para.lstrip()
    if not t or not t[0].islower():
        return None
    if CAPTION_RE.match(t) or re.match(r"^Table\b", t, re.IGNORECASE):
        return None
    m = _HYPHEN_CONTINUATION_PREFIX_RE.match(t)
    return (m.group(1) + (m.group(2) or "")) if m else None


def _extend_mention_text(para, page_idx, para_idx, paras, pages) -> str:
    out = para.strip()
    extensions = 0
    for nxt in _continuation_paragraphs(page_idx, para_idx, paras, pages):
        if extensions >= MAX_MENTION_EXTENSIONS:
            break
        nxt_text = nxt.strip()
        if not nxt_text:
            continue
        if out.rstrip().endswith("-"):
            merged_base = out.rstrip()[:-1]
            if _is_safe_continuation(nxt_text):
                out = _collapse_ws(merged_base + nxt_text)
                extensions += 1
                continue
            prefix = _extract_hyphen_continuation_prefix(nxt_text)
            if prefix:
                out = _collapse_ws(merged_base + prefix)
            break
        if _ends_complete_sentence(out) or out.rstrip().endswith(":"):
            break
        if not _is_safe_continuation(nxt_text):
            continue
        if extensions == 0 and not nxt_text[0].islower():
            break
        out = _collapse_ws(out + " " + nxt_text)
        extensions += 1
    return out


def _is_usable_mention(text: str) -> bool:
    t = text.strip()
    if len(t) < MIN_MENTION_CHARS or re.fullmatch(r"-+", t):
        return False
    if _HEADING_ONLY_RE.match(t) and len(t) < 80:
        return False
    if t.startswith("$$") or t.startswith("\\["):
        return False
    url_hits = len(_FOOTNOTE_URL_RE.findall(t))
    if url_hits >= 2 or (url_hits >= 1 and len(t) < 200):
        return False
    return True


def find_mentions(pages: List[str]) -> Dict[str, List[dict]]:
    mentions: Dict[str, List[dict]] = {}
    seen: Dict[str, set] = {}
    for page_idx, page_text in enumerate(pages, start=1):
        paras = page_paragraphs_without_tables(page_text)
        for i, para in enumerate(paras):
            if CAPTION_RE.match(para):
                continue
            nums = _table_nums_in_para(para)
            if not nums:
                continue
            lo = max(0, i - ADJACENT_PARA_WINDOW)
            hi = min(len(paras), i + ADJACENT_PARA_WINDOW + 1)
            for j in range(lo, hi):
                candidate = paras[j]
                if CAPTION_RE.match(candidate):
                    continue
                if j != i:
                    adj_nums = _table_nums_in_para(candidate)
                    if adj_nums and adj_nums.isdisjoint(nums):
                        continue
                text = _extend_mention_text(candidate, page_idx, j, paras, pages)
                if not _is_usable_mention(text):
                    continue
                for n in nums:
                    key = (page_idx, text)
                    seen.setdefault(n, set())
                    if key in seen[n]:
                        continue
                    seen[n].add(key)
                    mentions.setdefault(n, []).append({"page": page_idx, "text": text})
    for n in mentions:
        mentions[n].sort(key=lambda m: (m["page"], m["text"]))
    return mentions

print("Caption/mention heuristics ready")


In [ ]:
# --- GLiNER2 datasets/metrics from table HTML ---

GLINER2_LABEL_DESCRIPTIONS = {
    "model": "Name of a model, method, or algorithm (e.g. TransE, ComplEx, RotatE).",
    "dataset": "Name of a benchmark dataset or knowledge-graph corpus (e.g. WN18, FB15k, YAGO).",
    "metric": "Name of an evaluation metric used for reporting performance (e.g. MRR, Hits@10, F1).",
}
GLINER2_LABELS = list(GLINER2_LABEL_DESCRIPTIONS.keys())


def _strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def normalize_text(text: object) -> str:
    if text is None:
        return ""
    s = _strip_accents(str(text).strip().lower())
    return re.sub(r"\s+", " ", s)


def normalize_dataset(text: object) -> str:
    return normalize_text(text).replace(" ", "").replace("_", "").replace("-", "")


def _strip_trailing_plural(s: str) -> str:
    if not s or "@" in s or not s.isalpha():
        return s
    if len(s) > 3 and s.endswith("s") and not s.endswith("ss"):
        return s[:-1]
    return s


def normalize_metric(text: object) -> str:
    s = normalize_text(text)
    if not s:
        return ""
    return _strip_trailing_plural(re.sub(r"[^a-z0-9@]+", "", s))


def metric_dedup_key(raw: str) -> str:
    nm = normalize_metric(raw)
    if not nm:
        return ""
    m = re.match(r"^h@(\d+)$", nm)
    return f"hits@{m.group(1)}" if m else nm


def _is_incomplete_metric_key(key: str) -> bool:
    if not key or "@" not in key:
        return False
    suffix = key.split("@", 1)[1]
    return not suffix or not any(ch.isdigit() for ch in suffix)


def _dedup_by_key(items: List[str], key_fn) -> List[str]:
    buckets: Dict[str, List[str]] = {}
    for value in items:
        key = key_fn(value)
        if not key:
            continue
        buckets.setdefault(key, []).append(value)
    return sorted(
        max(vals, key=lambda s: (len(s), any(c.isupper() for c in s)))
        for vals in buckets.values()
    )


def header_and_rows_from_html(html: str) -> Tuple[List[str], List[str]]:
    soup = BeautifulSoup(html, "html.parser")
    thead = soup.find("thead")
    tbody = soup.find("tbody")

    def _row_text(tr):
        cells = [c.get_text(separator=" ", strip=True) for c in tr.find_all(["th", "td"])]
        cells = [c for c in cells if c]
        return " | ".join(cells) if cells else None

    header_lines, body_lines = [], []
    if thead is not None:
        for tr in thead.find_all("tr"):
            txt = _row_text(tr)
            if txt:
                header_lines.append(txt)
    trs = tbody.find_all("tr") if tbody is not None else soup.find_all("tr")
    for tr in trs:
        if thead is not None and tr in thead.find_all("tr"):
            continue
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue
        txt = _row_text(tr)
        if not txt:
            continue
        if all(c.name == "th" for c in cells) and not body_lines:
            header_lines.append(txt)
        else:
            body_lines.append(txt)
    if not header_lines and body_lines:
        header_lines, body_lines = [body_lines[0]], body_lines[1:]
    return header_lines, body_lines


def _clean_entity(value: str) -> str:
    s = str(value).strip()
    s = re.sub(r"\[[^\]]{1,50}\]", "", s)
    s = re.sub(r"\((?:[^\)]*\d{4}[^\)]*|[^\)]*et\s*al\.?[^\)]*)\)", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\$([^$]+?)\$", r"\1", s)
    s = re.sub(r"\\(?:textbf|textit|text|mathbf|mathrm|mathit)\{([^{}]*)\}", r"\1", s)
    for _ in range(3):
        s = re.sub(r"[_^]\{([^{}]*)\}", r"\1", s)
    s = s.replace("{", "").replace("}", "")
    return re.sub(r"\s+", " ", s).strip(" .,:;-")


def _extract_entities_raw(text: str) -> Dict[str, Tuple[str, float]]:
    best: Dict[str, Tuple[str, float]] = {}
    if not text or not text.strip():
        return best
    try:
        result = gliner2_model.extract_entities(
            text[:GLINER2_MAX_CHARS],
            GLINER2_LABEL_DESCRIPTIONS,
            include_confidence=True,
        )
    except Exception as e:
        print(f"  [gliner2 error] {type(e).__name__}: {e}")
        return best
    for label, items in (result or {}).get("entities", {}).items():
        if label not in GLINER2_LABELS:
            continue
        for item in items or []:
            if isinstance(item, dict):
                raw, score = str(item.get("text", "")), float(item.get("confidence", 1.0) or 1.0)
            else:
                raw, score = str(item), 1.0
            value = _clean_entity(raw)
            if score < GLINER2_MIN_SCORE or len(value) < 2:
                continue
            if re.fullmatch(r"[+-]?\d+(?:\.\d+)?", value):
                continue
            prev = best.get(value)
            if prev is None or score > prev[1]:
                best[value] = (label, score)
    return best


def _merge_best(target, other):
    for value, (label, score) in other.items():
        prev = target.get(value)
        if prev is None or score > prev[1]:
            target[value] = (label, score)


def _keys_from_best(best, label, key_fn):
    return {key_fn(v) for v, (lbl, _) in best.items() if lbl == label and key_fn(v)}


def extract_table_entities(caption: str, html: str) -> Dict[str, List[str]]:
    header_lines, body_lines = header_and_rows_from_html(html)
    header_context = "\n".join(header_lines)
    table_best: Dict[str, Tuple[str, float]] = {}
    if header_context:
        _merge_best(table_best, _extract_entities_raw(header_context))
    for row_text in body_lines:
        prompt = f"Table column headers: {header_context}\nRow: {row_text}" if header_context else row_text
        _merge_best(table_best, _extract_entities_raw(prompt))
    if not table_best:
        _merge_best(table_best, _extract_entities_raw("\n".join(header_lines + body_lines)))

    caption_only_ds = caption_only_mt = set()
    if caption:
        cap_best = _extract_entities_raw(caption)
        caption_only_ds = _keys_from_best(cap_best, "dataset", normalize_dataset) - _keys_from_best(table_best, "dataset", normalize_dataset)
        caption_only_mt = _keys_from_best(cap_best, "metric", metric_dedup_key) - _keys_from_best(table_best, "metric", metric_dedup_key)

    datasets, metrics = [], []
    for value, (label, _) in table_best.items():
        if label == "dataset":
            k = normalize_dataset(value)
            if k and k not in caption_only_ds:
                datasets.append(value)
        elif label == "metric":
            k = metric_dedup_key(value)
            if k and not _is_incomplete_metric_key(k) and k not in caption_only_mt:
                metrics.append(value)
    return {
        "dataset": _dedup_by_key(datasets, normalize_dataset),
        "metric": _dedup_by_key(metrics, metric_dedup_key),
    }

print("GLiNER helpers ready")


In [ ]:
# --- LaTeX cleanup + per-PDF extraction ---

_LATEX_MATH_RE = re.compile(r"\$\$([^$]+)\$\$|\$([^$]+)\$|\\\(([^)]+)\\\)|\\\[([^\]]+)\\\]")
_latex2text = None


def _get_latex2text():
    global _latex2text
    if _latex2text is None:
        from pylatexenc.latex2text import LatexNodes2Text
        _latex2text = LatexNodes2Text()
    return _latex2text


def latex_to_plain(text: str) -> str:
    if not text or not str(text).strip():
        return text

    def _repl(match: re.Match) -> str:
        fragment = next(g for g in match.groups() if g is not None)
        try:
            return _get_latex2text().latex_to_text(fragment)
        except Exception:
            return fragment

    return _LATEX_MATH_RE.sub(_repl, str(text))


def format_mentions(mentions: List[dict]) -> List[dict]:
    return [{"page": m["page"], "text": latex_to_plain(m["text"])} for m in mentions]


def extract_gt_for_pdf(pdf_path: Path, *, force_ocr: bool = False) -> dict:
    pages = ocr_pdf_pages(pdf_path, force_ocr=force_ocr)
    mentions_by_num = find_mentions(pages)
    tables = []
    for page_idx, page_text in enumerate(pages, start=1):
        paired = pair_captions_to_tables(page_text)
        for t_i, item in enumerate(paired, start=1):
            html = item["match"].group(0)
            caption_raw = item["caption"] or ""
            table_num = item["table_num"]
            mentions_raw = mentions_by_num.get(table_num, []) if table_num else []
            ents = extract_table_entities(caption_raw, html)
            tables.append({
                "table_id": f"{pdf_path.stem}_p{page_idx}_t{t_i}",
                "table_label": f"Table {table_num}" if table_num else None,
                "page": page_idx,
                "caption": latex_to_plain(caption_raw),
                "mentions": format_mentions(mentions_raw),
                "datasets": ents["dataset"],
                "metrics": ents["metric"],
            })
    return {
        "paper": pdf_path.stem,
        "source_pdf": f"{PDF_DIR.as_posix().rstrip('/')}/{pdf_path.name}",
        "num_tables": len(tables),
        "tables": tables,
    }

print("Extraction helpers ready")


In [ ]:
# --- Build ground_truth_context.json ---
documents = []
for pdf_path in PDF_FILES:
    print("\n" + "=" * 70)
    print(f"Processing: {pdf_path.name}")
    print("=" * 70)
    doc = extract_gt_for_pdf(pdf_path, force_ocr=FORCE_OCR)
    documents.append(doc)
    n_cap = sum(1 for t in doc["tables"] if t["caption"])
    n_men = sum(1 for t in doc["tables"] if t["mentions"])
    print(
        f"  tables={doc['num_tables']} | with_caption={n_cap} | with_mentions={n_men} | "
        f"datasets={sum(len(t['datasets']) for t in doc['tables'])} | "
        f"metrics={sum(len(t['metrics']) for t in doc['tables'])}"
    )

payload = {"version": "1.0", "documents": documents}
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)
print(f"Saved: {OUTPUT_PATH.resolve()}")
print(f"documents: {len(documents)} | tables: {sum(d['num_tables'] for d in documents)}")
print("Review/edit this JSON before using it as gold ground truth.")


In [ ]:
# Preview first document / first table
preview = json.loads(OUTPUT_PATH.read_text(encoding="utf-8"))
doc0 = preview["documents"][0]
t0 = doc0["tables"][0] if doc0["tables"] else {}
print("paper:", doc0.get("paper"))
print("num_tables:", doc0.get("num_tables"))
if t0:
    print("table_id:", t0.get("table_id"))
    print("table_label:", t0.get("table_label"))
    print("caption:", (t0.get("caption") or "")[:160])
    print("n_mentions:", len(t0.get("mentions") or []))
    print("datasets:", t0.get("datasets"))
    print("metrics:", t0.get("metrics"))
